# Filter Google Local Washington data to the Seattle area

Find Seattle-area restaurants in the metadata, keep only their reviews, and attach the business location to each review.

In [16]:
import pandas as pd

## 1. Seattle-area restaurants
Filter based on `, <City>, WA <zip>` suffix for the cities in `CITIES`. Filtering for restaurants based on if any of the categories contains "restaurant" (ex: `Restaurant`, `Chinese restaurant`, `Takeout Restaurant`).

In [17]:
CITIES = ["Seattle", "Bellevue", "Redmond", "Shoreline", "Lynnwood", "Bothell", "Kirkland",
          "Renton", "Kent", "Tukwila", "Edmonds", "Federal Way"]

meta = pd.read_json("data/meta-Washington.json", lines=True)
meta["city"] = meta["address"].str.extract(r", (" + "|".join(CITIES) + r"), WA \d{5}", expand=False)
in_seattle = meta["city"].notna()
is_restaurant = meta["category"].apply(lambda cats: isinstance(cats, list) and any("restaurant" in c.lower() for c in cats))
seattle_meta = meta[in_seattle & is_restaurant].drop_duplicates("gmap_id")
seattle_ids = set(seattle_meta["gmap_id"])

seattle_meta.to_json("data/meta-SeattleArea.json", orient="records", lines=True)
print(f"{len(seattle_ids):,} Seattle-area restaurants out of {meta['gmap_id'].nunique():,} businesses")
print(seattle_meta["city"].value_counts())
seattle_meta.head()

5,816 Seattle-area restaurants out of 120,642 businesses
city
Seattle        3217
Bellevue        487
Kent            323
Renton          277
Lynnwood        274
Kirkland        252
Federal Way     238
Redmond         221
Bothell         162
Tukwila         154
Edmonds         123
Shoreline        88
Name: count, dtype: int64


,name,address,gmap_id,description,latitude,longitude,category,avg_rating,num_of_reviews,price,hours,MISC,state,relative_results,url,city
129,Ed's Kort Haus,"Ed's Kort Haus, 6732 Greenwood Ave N, Seattle,...",0x5490143a1a6be1a3:0xcdfc8b297798e8f2,None,47.678611,-122.355000,[Hamburger restaurant],4.0,14,None,"[[Wednesday, 2PM–2AM], [Thursday, 2PM–2AM], [F...","{'Service options': ['Delivery'], 'Highlights'...",Permanently closed,"[0x5490143b3a099223:0xfe6baadd7d3f8ff2, 0x5490...",https://www.google.com/maps/place//data=!4m2!3...,Seattle
255,Kim's Kitchen,"Kim's Kitchen, Above Palace Spa, 1727 S 316th ...",0x549057a7db40d70d:0xe085b6f6463ca410,None,47.318445,-122.311246,[Korean restaurant],4.7,13,None,"[[Wednesday, 11AM–8PM], [Thursday, 11AM–8PM], ...","{'Service options': ['Delivery', 'Takeout'], '...",Permanently closed,None,https://www.google.com/maps/place//data=!4m2!3...,Federal Way
374,Baguette Vietnamese Sandwiches,"Baguette Vietnamese Sandwiches, 1509 E Madison...",0x54906ad1f83f2271:0xace5d4d692669216,None,47.614497,-122.312126,"[Restaurant, Sandwich shop, Vietnamese restaur...",5.0,1,None,"[[Wednesday, 11AM–3PM], [Thursday, 11AM–3PM], ...","{'Service options': ['Delivery'], 'Accessibili...",Permanently closed,"[0x5490148ada4ba629:0x75027ce1c1eb092c, 0x5490...",https://www.google.com/maps/place//data=!4m2!3...,Seattle
415,Grand Peking Restaurant,"Grand Peking Restaurant, 8075 161st Avenue Nor...",0x549072ad84235701:0xa7a6e17f497f3a39,Casual eatery offering classic Chinese dishes ...,47.675703,-122.126309,"[Chinese restaurant, Asian restaurant, Karaoke...",3.1,46,$$,"[[Wednesday, 10:30AM–9PM], [Thursday, 10:30AM–...","{'Service options': ['Takeout', 'Delivery'], '...",Permanently closed,"[0x54906d00b5eb0b93:0x56c65de8adbf1c0d, 0x5490...",https://www.google.com/maps/place//data=!4m2!3...,Redmond
429,Citizen Campfire Beer Garden,"Citizen Campfire Beer Garden, 706 Taylor Ave N...",0x549015dcda35c8b5:0x59acf0fabf61e550,None,47.625712,-122.345954,"[Bar & grill, Bar, Restaurant]",4.0,7,None,None,"{'Service options': ['Curbside pickup', 'No-co...",None,None,https://www.google.com/maps/place//data=!4m2!3...,Seattle


## 2. Seattle-area reviews

In [18]:
chunks = pd.read_json("data/review-Washington.json", lines=True, chunksize=500_000, dtype={"user_id": str})
reviews = pd.concat(c[c["gmap_id"].isin(seattle_ids)] for c in chunks)
print(f"{len(reviews):,} Seattle-area reviews")

2,154,527 Seattle-area reviews


## 3. Attach business location to each review

In [19]:
business_cols = seattle_meta[["gmap_id", "name", "address", "city", "latitude", "longitude", "category"]]
reviews = reviews.merge(business_cols, on="gmap_id", suffixes=("", "_business"))

reviews.to_json("data/review-SeattleArea.json", orient="records", lines=True)
reviews.head()

,user_id,name,time,rating,text,pics,resp,gmap_id,name_business,address,city,latitude,longitude,category
0,114851985077823750062,Joshua McCormack,1476746560454,5.0,I feel like not a lot of people know about thi...,None,None,0x5490143a1a6be1a3:0xcdfc8b297798e8f2,Ed's Kort Haus,"Ed's Kort Haus, 6732 Greenwood Ave N, Seattle,...",Seattle,47.678611,-122.355,[Hamburger restaurant]
1,110852791995453216926,Bettina Graupe,1388485302766,4.0,"If you want upscale or pretentious, look elsew...",None,None,0x5490143a1a6be1a3:0xcdfc8b297798e8f2,Ed's Kort Haus,"Ed's Kort Haus, 6732 Greenwood Ave N, Seattle,...",Seattle,47.678611,-122.355,[Hamburger restaurant]
2,105521314408384917549,J V,1413911526424,5.0,"It's a dive bar with great service, Do not exp...",None,None,0x5490143a1a6be1a3:0xcdfc8b297798e8f2,Ed's Kort Haus,"Ed's Kort Haus, 6732 Greenwood Ave N, Seattle,...",Seattle,47.678611,-122.355,[Hamburger restaurant]
3,113263257499923499987,Rachel Breitenbucher,1439148033917,5.0,Had a great experience last night visiting Ed'...,None,None,0x5490143a1a6be1a3:0xcdfc8b297798e8f2,Ed's Kort Haus,"Ed's Kort Haus, 6732 Greenwood Ave N, Seattle,...",Seattle,47.678611,-122.355,[Hamburger restaurant]
4,112650185984208886011,Naomi Ishikawa,1325365056929,4.0,"Service is good. Food's alright, but...BUT......",None,None,0x5490143a1a6be1a3:0xcdfc8b297798e8f2,Ed's Kort Haus,"Ed's Kort Haus, 6732 Greenwood Ave N, Seattle,...",Seattle,47.678611,-122.355,[Hamburger restaurant]
